# TIP/IEF 市场差：Stage1 现金优势 H5 信息门

冻结原39项、单一新增坐标、训练均值及两项匹配比较；当前回溯来源只作开发反馈。

In [1]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_tip_ief_cash_h5_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal']
feature_card=root/src['feature_card'];original_card=root/src['original39_dataset_card']
assert sha(feature_card/'summary.json')==src['feature_summary_sha256'] and sha(original_card/'summary.json')==src['original39_summary_sha256']
assert sha(root/'src/analysis/seed_disagreement_information.py')==src['bootstrap_module_sha256']
source_artifact=feature_card/'qlib_recorder'/src['feature_experiment_id']/src['feature_recorder_id']/'artifacts/analysis/source_map';dataset_artifact=original_card/'qlib_recorder'/src['original39_experiment_id']/src['original39_recorder_id']/'artifacts/inputs/dataset'
assert sha(source_artifact)==src['feature_artifact_sha256'] and sha(dataset_artifact)==src['original39_dataset_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(feature_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['feature_experiment_id'],recorder_id=src['feature_recorder_id']);assert rec.status=='FINISHED';source_map=rec.load_object('analysis/source_map')
with R.uri_context(str(original_card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['original39_experiment_id'],recorder_id=src['original39_recorder_id']);assert rec.status=='FINISHED';dataset=rec.load_object('inputs/dataset')
coordinate=source_map.set_index('signal_date').tip_minus_ief_5session_log_return_tminus2
assert coordinate.index.is_unique and source_map.known.sum()>=3964
tr_all=dataset.prepare('train',col_set=['feature','label','meta']);he_all=dataset.prepare('held',col_set=['feature','label','meta']);names=list(tr_all['feature'].columns);assert len(names)==39
tr_dates=pd.DatetimeIndex(tr_all.index.get_level_values('datetime')).normalize();he_dates=pd.DatetimeIndex(he_all.index.get_level_values('datetime')).normalize();tr_feature=coordinate.reindex(tr_dates).to_numpy(float);he_feature=coordinate.reindex(he_dates).to_numpy(float)
tr_mask=tr_all[('meta','train_mature')].to_numpy(bool)&np.isfinite(tr_feature);he_mask=he_all[('meta','held_mature')].to_numpy(bool)&np.isfinite(he_feature)
tr=tr_all.iloc[np.flatnonzero(tr_mask)];he=he_all.iloc[np.flatnonzero(he_mask)];ct=tr_feature[tr_mask];ch=he_feature[he_mask]
expected=plan['cohorts'];assert len(tr)==expected['expected_train_mature_rows']==3014 and len(he)==expected['expected_held_mature_rows']==940 and len(he_all)==expected['full_held_calendar_rows']==944
xtr=tr['feature'][names].to_numpy(float);xhe=he['feature'][names].to_numpy(float);xtr_new=np.column_stack([xtr,ct]);xhe_new=np.column_stack([xhe,ch]);ytr=tr[('label','cash_advantage_H5')].to_numpy(float);yhe=he[('label','cash_advantage_H5')].to_numpy(float);assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
preflight={'status':'matched_original39_and_TIP_IEF_source_known_cash_H5_preflight_passed','plan_sha256':sha(card/'plan.json'),'train_mature_source_known':len(tr),'held_mature_source_known':len(he),'full_held_calendar':len(he_all),'source_Recorder_object_loads':2,'model_fits':0,'native_accounts':0};(card/'preflight_result.json').write_text(json.dumps(preflight,indent=2)+'\n')
state={'status':'fit_started','plan_sha256':preflight['plan_sha256'],'fit_attempts':0,'fit_completed':0,'bootstrap_attempts':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
base=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));candidate=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
for model,x in ((base,xtr),(candidate,xtr_new)):
 state['fit_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');model.fit(x,ytr);state['fit_completed']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
p0=base.predict(xhe);p1=candidate.predict(xhe_new);mean=float(ytr.mean());coef=float(candidate.named_steps['ridge'].coef_[-1])
scored=pd.DataFrame({'signal_date':pd.DatetimeIndex(he.index.get_level_values('datetime')),'target':yhe,'original39':p0,'candidate':p1,'train_mean':mean})
scored['improvement_vs_original39']=(yhe-p0)**2-(yhe-p1)**2;scored['improvement_vs_train_mean']=(yhe-mean)**2-(yhe-p1)**2
full=pd.DataFrame({'signal_date':pd.DatetimeIndex(he_all.index.get_level_values('datetime'))}).merge(scored,on='signal_date',how='left',validate='one_to_one');assert len(full)==944 and int(full.candidate.notna().sum())==940
metrics={'candidate_held_mse':float(np.mean((yhe-p1)**2)),'original39_held_mse':float(np.mean((yhe-p0)**2)),'train_mean_held_mse':float(np.mean((yhe-mean)**2)),'train_target_mean':mean,'added_standardized_coefficient':coef,'comparators':{}};gates={'coefficient_positive':bool(coef>0)};replicates={};indices=None
for key in ('original39','train_mean'):
 column='improvement_vs_'+key;diff=scored[column]
 halves=[]
 for start,end in plan['cohorts']['held_halves']:
  z=scored.loc[scored.signal_date.between(start,end),column];halves.append({'start':start,'end':end,'rows':len(z),'mean_improvement':float(z.mean())})
 annual=[]
 for year in plan['cohorts']['complete_held_years']:
  z=scored.loc[scored.signal_date.dt.year.eq(year),column];annual.append({'year':year,'rows':len(z),'mean_improvement':float(z.mean())})
 state['bootstrap_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 b=plan['information_gate']['bootstrap'];interval,idx,reps=full_calendar_bootstrap(full[column].to_numpy(float),mean_block=b['mean_block_sessions'],resamples=b['draws_each'],seed=b['seed'],alpha=b['one_sided_alpha_each']);assert interval['valid_replicates']==b['draws_each']
 if indices is None:indices=idx
 else:assert np.array_equal(indices,idx)
 replicates[key]=reps;metrics['comparators'][key]={'full_mean_improvement':float(diff.mean()),'halves':halves,'complete_years':annual,'bootstrap':interval}
 gates[key]={'full_positive':bool(diff.mean()>0),'both_halves_positive':bool(all(x['mean_improvement']>0 for x in halves)),'both_complete_years_positive':bool(all(x['mean_improvement']>0 for x in annual)),'corrected_lower_positive':bool(interval['lower_bound']>0)}
pass_gate=bool(gates['coefficient_positive'] and all(all(v.values()) for k,v in gates.items() if isinstance(v,dict)))
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,preflight=preflight)
 rec.save_objects(artifact_path='models',original39=base,candidate=candidate)
 rec.save_objects(artifact_path='predictions',scored=scored)
 rec.save_objects(artifact_path='analysis',full_held_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=indices,information_gate_pass=pass_gate)
 rec.log_metrics(model_fits=state['fit_completed'],native_accounts=0,information_gate_pass=int(pass_gate),improvement_vs_original39=metrics['comparators']['original39']['full_mean_improvement'],improvement_vs_train_mean=metrics['comparators']['train_mean']['full_mean_improvement'])
 rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def ready(x):
 if isinstance(x,dict):return {str(k):ready(v) for k,v in x.items()}
 if isinstance(x,(list,tuple)):return [ready(v) for v in x]
 if isinstance(x,np.generic):return x.item()
 return x
summary={'experiment':plan['experiment'],'decision':'conditional_TIP_IEF_cash_H5_information_pass_source_PIT_pending' if pass_gate else 'not_promoted_fixed_TIP_IEF_cash_H5_information_failed','objective_achieved':False,'plan_sha256':preflight['plan_sha256'],'preflight':preflight,'metrics':metrics,'gates':gates,'information_gate_pass':pass_gate,'new_model_fits':state['fit_completed'],'new_bootstraps':state['bootstrap_attempts'],'new_native_accounts':0,'held_history_reused':True,'historical_first_public_qualified':False,'independent_forward_qualified_days':0,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['Conditional current-vintage issuer source has unknown 2016-2022 revisions and daily first-public timing.','Information MSE is not a fee-net funded Stage1 account.','Repeated 2023-2026 history is development only, not independent forward evidence.']};summary=ready(summary);(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];summary['NAS_evidence']=evidence;summary['NAS_readback']={'sha256':remote,'verified':True};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'preflight':preflight,'metrics':metrics,'gates':gates,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False,indent=2))


{
  "decision": "not_promoted_fixed_TIP_IEF_cash_H5_information_failed",
  "preflight": {
    "status": "matched_original39_and_TIP_IEF_source_known_cash_H5_preflight_passed",
    "plan_sha256": "7731a593b27ac0a776c945c11d7936247302eb49a2a14de59bdb868b30a303ab",
    "train_mature_source_known": 3014,
    "held_mature_source_known": 940,
    "full_held_calendar": 944,
    "source_Recorder_object_loads": 2,
    "model_fits": 0,
    "native_accounts": 0
  },
  "metrics": {
    "candidate_held_mse": 0.0025280546087696003,
    "original39_held_mse": 0.0025262669365982205,
    "train_mean_held_mse": 0.0023466841316487535,
    "train_target_mean": -0.0032255746731863185,
    "added_standardized_coefficient": -0.0012220505371025217,
    "comparators": {
      "original39": {
        "full_mean_improvement": -1.7876721713794678e-06,
        "halves": [
          {
            "start": "2023-01-03",
            "end": "2024-12-31",
            "rows": 502,
            "mean_improvement": -1.1602

## 中文结论

固定 TIP−IEF t−6..t−2 现金 H5 信息门失败。训练3014、评价成熟940；新增系数−0.00122205与预定正号相反，候选评价MSE0.002528055劣于原39项0.002526267和训练均值0.002346684；完整和区块下界均未通过。2拟合、2区块检验、0账户。Recorder与NAS选定工件冷读一致；不改符号、ETF或窗口重试。